# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their available fields
print("Available Record Sets:\n---------------------")
record_sets = []
for record_set in dataset.metadata.record_sets:
    record_sets.append(record_set['@id'])
    print(f"Record Set @id: {record_set['@id']}")
    print(f"    Name: {record_set.get('name', 'N/A')}")
    print(f"    Description: {record_set.get('description', 'N/A')}")
    print("    Fields:")
    for field in record_set.get('field', []):
        print(f"        Field @id: {field['@id']} | name: {field.get('name','N/A')}")
    print()

print(f"Record set @ids: {record_sets}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Here we extract all available record sets into DataFrames (if more, append their @id here)
# For this dataset, let's first list all record sets. If only one main, we'll use it.
record_sets_ids = record_sets # from previous cell
dataframes = {}

for record_set_id in record_sets_ids:
    # Retrieve records for each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for: {record_set_id}, columns: {dataframes[record_set_id].columns.tolist()}")
    else:
        print(f"No records found for record set: {record_set_id}")

# For demonstration, pick the first record set
if record_sets_ids:
    main_record_set = record_sets_ids[0]
    if main_record_set in dataframes and not dataframes[main_record_set].empty:
        print(dataframes[main_record_set].head())
    else:
        print("Main record set is empty.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis
# You may adjust these @id variables based on the printed columns and schema above.

if record_sets_ids:
    df = dataframes.get(main_record_set, pd.DataFrame())
    if not df.empty:
        print("Available columns:", df.columns.tolist())
        # Let's assume a common field for EDA, e.g., 'cr:field:mw' (Molecular Weight), if available
        possible_numeric_fields = [col for col in df.columns if any(substr in col.lower() for substr in ['mw', 'weight', 'count', 'abundance', 'coverage', 'length', 'quantity', 'peptide'])]
        
        if possible_numeric_fields:
            numeric_field = possible_numeric_fields[0]
            print(f"Using numeric field: {numeric_field}")
            # Filter rows with field value > threshold (adjust threshold as suitable for the specific field)
            threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
            
            # Handle conversion if field is not numeric
            if not pd.api.types.is_numeric_dtype(df[numeric_field]):
                df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold}:")
            print(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Grouping by a field (e.g., 'cr:field:accession' or 'cr:field:description')
            possible_group_fields = [col for col in df.columns if any(substr in col.lower() for substr in ['sample', 'type', 'accession', 'category', 'description'])]
            if possible_group_fields:
                group_field = possible_group_fields[0]
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by {group_field}:")
                print(grouped_df.head())
            else:
                print("No appropriate group field found for grouping.")
        else:
            print("No numeric fields detected for analysis.")
    else:
        print("Main DataFrame is empty.")
else:
    print("No record sets found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets_ids:
    df = dataframes.get(main_record_set, pd.DataFrame())

    # Plot distribution of the selected numeric field after normalization (if filtered_df available)
    if 'filtered_df' in locals() and not filtered_df.empty and f'{numeric_field}_normalized' in filtered_df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(filtered_df[f'{numeric_field}_normalized'], kde=True)
        plt.title(f'Distribution of Normalized {numeric_field}')
        plt.xlabel(f'Normalized {numeric_field}')
        plt.ylabel('Count')
        plt.show()

    # Plot relationship between numeric_field and group_field, if available
    if 'possible_group_fields' in locals() and possible_group_fields:
        group_field = possible_group_fields[0]
        if group_field in filtered_df.columns:
            plt.figure(figsize=(12, 6))
            sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
            plt.title(f'{numeric_field} by {group_field}')
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using the `mlcroissant` library, we loaded and explored the Mass Spectrometry Analysis of Extracellular Vesicles dataset.
- We listed record sets and examined their structure using `@id` fields, showcasing reproducible querying using Croissant schema conventions.
- The extracted DataFrame enabled basic filtering, normalization, and grouping by selected features.
- Visualization highlighted the distribution and categorical breakdown of a key numeric property from the dataset.
<br><br>
This notebook can be easily extended by selecting other record sets or fields via their `@id` and applying additional domain-specific analyses.